# Offline Policy Evaluation


### Introduction

This notebook demonstrates the use of offline policy evaluation for MABs.

### Objectives

#### Evaluation:

Evaluate the performance of a MAB using multiple offline policy estimators.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from pybandits.cmab import CmabBernoulliCC
from pybandits.offline_policy_evaluator import OfflinePolicyEvaluator

%load_ext autoreload
%autoreload 2

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate data

We first generate a binarly labeled data set, with a two dimensional feature space, and is not lineraly seprabale.
We then split the data set to a training data setm and a test data set.

In [2]:
n_samples = 1000
n_actions = 2
n_batches = 3
n_rewards = 1
n_groups = 2
n_features = 3

In [3]:
unique_actions = [f"a{i}" for i in range(n_actions)]
action_ids = np.random.choice(unique_actions, n_samples * n_batches)
batches = [i for i in range(n_batches) for _ in range(n_samples)]
rewards = [np.random.randint(2, size=(n_samples * n_batches)) for _ in range(n_rewards)]
action_true_rewards = {(a, r): np.random.rand() for a in unique_actions for r in range(n_rewards)}
true_rewards = [
    np.array([action_true_rewards[(a, r)] for a in action_ids]).reshape(n_samples * n_batches) for r in range(n_rewards)
]
groups = np.random.randint(n_groups, size=n_samples * n_batches)
action_costs = {action: np.random.rand() for action in unique_actions}
costs = np.array([action_costs[a] for a in action_ids])
context = np.random.rand(n_samples * n_batches, n_features)
action_propensity_score = {action: np.random.rand() for action in unique_actions}
propensity_score = np.array([action_propensity_score[a] for a in action_ids])
df = pd.DataFrame(
    {
        "batch": batches,
        "action_id": action_ids,
        "cost": costs,
        "group": groups,
        **{f"reward_{r}": rewards[r] for r in range(n_rewards)},
        **{f"true_reward_{r}": true_rewards[r] for r in range(n_rewards)},
        **{f"context_{i}": context[:, i] for i in range(n_features)},
        "propensity_score": propensity_score,
    }
)
contextual_features = [col for col in df.columns if col.startswith("context")]

## Generate Model

Using the cold_start method of CmabBernoulliCC, we can create a model to be used for offline policy evaluation.

In [4]:
action_ids_cost = {action_id: df["cost"][df["action_id"] == action_id].iloc[0] for action_id in unique_actions}

mab = CmabBernoulliCC.cold_start(action_ids_cost=action_ids_cost, n_features=len(contextual_features))

## OPE

Given the model and the OPE data from the logging policy, we can either evaluate the model using the logging policy, or update it with the logging policy data prior to the evaluation.

In [5]:
evaluator = OfflinePolicyEvaluator(
    logged_data=df,
    split_prop=0.5,
    n_trials=10,
    fast_fit=True,
    scaler=MinMaxScaler(),
    ope_estimators=None,
    verbose=True,
    propensity_score_model_type="batch_empirical",
    expected_reward_model_type="gbm",
    importance_weights_model_type="logreg",
    batch_feature="batch",
    action_feature="action_id",
    reward_feature="reward_0",
    true_reward_feature="true_reward_0",
    contextual_features=contextual_features,
    group_feature="group",
    cost_feature="cost",
    propensity_score_feature="propensity_score",
)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 266.45it/s]


2026-03-27 18:42:14.932 | INFO     | pybandits.offline_policy_evaluator:_estimate_propensity_score:853 - Data batch-empirical estimation of propensity score.


2026-03-27 18:42:14.940 | INFO     | pybandits.offline_policy_evaluator:_estimate_expected_reward:904 - Data prediction of expected reward based on gbm model.


In [6]:
evaluator.evaluate(mab=mab, visualize=True, n_mc_experiments=1000)

2026-03-27 18:42:15.248 | INFO     | pybandits.offline_policy_evaluator:estimate_policy:1001 - Data prediction of expected policy based on Monte Carlo experiments using 4 cores.


/opt/hostedtoolcache/Python/3.10.20/x64/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


  0%|          | 0/1000 [00:00<?, ?it/s]

2026-03-27 18:42:15.310 | INFO     | pybandits.offline_policy_evaluator:_mab_predict_serialized:1287 - Predicting actions for MC experiment 1.


2026-03-27 18:42:15.310 | INFO     | pybandits.offline_policy_evaluator:_mab_predict_serialized:1287 - Predicting actions for MC experiment 3.


...

2026-03-27 18:42:42.716 | INFO     | pybandits.offline_policy_evaluator:_estimate_importance_weight:943 - Data prediction of importance weights based on logreg model.


2026-03-27 18:42:42.776 | INFO     | pybandits.offline_policy_evaluator:evaluate:1089 - Offline Policy Evaluation for reward_0.


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/scipy/_lib/_util.py:440: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.509318,0.474781,0.544494,0.017496,b-ipw,reward_0
1,0.490722,0.485498,0.495929,0.002625,dm,reward_0
2,0.500461,0.468050,0.531992,0.016288,dr,reward_0
3,0.490722,0.485588,0.495958,0.002623,dros-opt,reward_0
4,0.500461,0.469165,0.532976,0.016308,dros-pess,reward_0
5,0.500256,0.468285,0.532346,0.016479,ipw,reward_0
6,0.000000,NaN,NaN,0.000000,rep,reward_0
7,0.500461,0.468006,0.533402,0.016397,sndr,reward_0
8,0.500231,0.468046,0.533052,0.016627,snips,reward_0
9,0.500461,0.468776,0.531756,0.016292,sg-dr,reward_0


In [7]:
evaluator.update_and_evaluate(mab=mab, visualize=True, n_mc_experiments=1000)

2026-03-27 18:42:43.819 | INFO     | pybandits.offline_policy_evaluator:_update_mab:1172 - Offline policy update for <class 'pybandits.cmab.CmabBernoulliCC'>.


SVI:   0%|          | 0/1000 [00:00<?, ?it/s]

SVI:   0%|          | 1/1000 [00:00<09:10,  1.81it/s]

SVI:   0%|          | 1/1000 [00:00<09:10,  1.81it/s, loss=7.3490]

SVI:   0%|          | 2/1000 [00:00<09:10,  1.81it/s, loss=6.7601]

...

2026-03-27 18:43:27.380 | INFO     | pybandits.offline_policy_evaluator:_estimate_importance_weight:943 - Data prediction of importance weights based on logreg model.


2026-03-27 18:43:27.442 | INFO     | pybandits.offline_policy_evaluator:evaluate:1089 - Offline Policy Evaluation for reward_0.


/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/scipy/_lib/_util.py:440: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


Loading BokehJS ...

,value,lower,upper,std,estimator,objective
0,0.511691,0.476084,0.548978,0.018774,b-ipw,reward_0
1,0.489672,0.484581,0.494859,0.002663,dm,reward_0
2,0.499714,0.465869,0.531604,0.016940,dr,reward_0
3,0.489672,0.484335,0.494978,0.002663,dros-opt,reward_0
4,0.499714,0.466709,0.533356,0.016944,dros-pess,reward_0
5,0.501143,0.467221,0.536594,0.017763,ipw,reward_0
6,0.000000,NaN,NaN,0.000000,rep,reward_0
7,0.499670,0.465854,0.532185,0.016777,sndr,reward_0
8,0.498977,0.464629,0.535256,0.018043,snips,reward_0
9,0.499714,0.466720,0.532568,0.017034,sg-dr,reward_0
